# Flux bounding and enumeration

**What's in this notebook?** This notebook introduces the `bounded_fluxes` class from `jaxvacua.flux_bounding`, which implements the systematic flux-bounding and enumeration algorithm of [arXiv:2501.03984](https://arxiv.org/abs/2501.03984). The algorithm provides provably complete enumeration of Type IIB SUSY flux vacua within a finite region of moduli space, subject to a D3-tadpole bound.

(*Created:* Andreas Schachner, 2025)

## Algorithm overview

The flux-bounding algorithm derives rigorous upper bounds on the integer flux quanta $(f, h)$ from the eigenvalues of two period-matrix-dependent objects:

- The **ISD matrix** $\mathcal{M}(z,\bar{z})$: its largest eigenvalue $\lambda_{\max}$ controls the size of the full flux vector.
- The **gauge kinetic matrix** $\mathcal{N}(z,\bar{z})$: the extreme eigenvalues of $-\mathrm{Im}(\mathcal{N})$ and $\mathrm{Im}(\mathcal{N}^{-1})$ bound the NSNS-flux sub-vectors $h_1, h_2$ individually.

Given a sample of moduli points, *global* extrema of these eigenvalues are computed and used to construct a bounding box for the integer NSNS-flux vectors $h = (h_1, h_2)$. A complete set of candidate $h$ vectors is then enumerated inside this box; for each candidate, an ISD-projected RR-flux $f$ is constructed via
$$
    f \approx \bigl(s\,\mathcal{M}\,\Sigma + c_0\bigr)\,h,
$$
and the resulting flux configuration is tested against the D3-tadpole constraint and all local eigenvalue bounds.

## Imports

In [ ]:
# General imports
import warnings
import numpy as np

# JAX imports
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# JAXVacua
import jaxvacua as jvc
from jaxvacua.flux_bounding import bounded_fluxes

warnings.filterwarnings('ignore')

## Model and sampler setup

We work with the Kreuzer–Skarke geometry at $h^{1,2} = 2$, which gives a flux lattice of dimension $b_3 = 2(h^{1,2}+1) = 6$ and a full flux vector of length 12. We choose an orientifold charge $Q_O = 276$.

The model is initialised from pre-computed period data stored in `jaxvacua/models/`:

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(h12=h12,model_ID=1)
print(model)

The `data_sampler` handles moduli and flux sampling. Here we restrict the imaginary parts of the complex structure moduli to $[2, 3]$, the axion $c_0 = \mathrm{Re}(\tau)$ to $[-0.5, 0.5]$, and the dilaton $s = \mathrm{Im}(\tau)$ to $[2, 5]$:

In [ ]:
sampler = jvc.data_sampler(
    model,
    moduli_bounds=(2., 3.),
    axion_bounds=(-0.5, 0.5),
    dilaton_bounds=(2., 5.),
)
print(sampler)

## Initialising `bounded_fluxes`

The `bounded_fluxes` class is the central object. It takes a model, an optional sampler, a D3-tadpole bound `Nmax`, and an optional lower bound `dil_min` on $s = \mathrm{Im}(\tau)$.

If `Nmax` is not supplied it defaults to `model.D3_tadpole`. If `dil_min` is not supplied it defaults to $\sqrt{3}/2$, the boundary of the $\mathrm{SL}(2,\mathbb{Z})$ fundamental domain.

In [ ]:
bf = bounded_fluxes(model, sampler=sampler, Nmax=50)
print(bf)
print(f"n_fluxes     = {bf.n_fluxes}   (= 2*(h12+1))")
print(f"dimension_H3 = {bf.dimension_H3}   (= h12+1)")
print(f"Nmax         = {bf.Nmax}")
print(f"dil_min      = {bf.dil_min:.4f}  (= sqrt(3)/2)")

## Eigenvalue quantities at a single moduli point

The method `compute_evs(moduli)` computes the five key eigenvalue quantities at a single point in moduli space:

| Quantity | Definition |
|---|---|
| $\lambda_{\max}$ | Largest eigenvalue of $\mathcal{M}$ (ISD matrix) |
| $\mu_{\min}$ | Smallest eigenvalue of $-\mathrm{Im}(\mathcal{N})$ |
| $\mu_{\max}$ | Largest eigenvalue of $-\mathrm{Im}(\mathcal{N})$ |
| $\tilde\mu_{\min}$ | Smallest eigenvalue of $\mathrm{Im}(\mathcal{N}^{-1})$ |
| $\tilde\mu_{\max}$ | Largest eigenvalue of $\mathrm{Im}(\mathcal{N}^{-1})$ |

This method is JIT-compiled and fast to evaluate repeatedly:

In [ ]:
# A test point inside the LCS region
moduli_test = jnp.array([0.1 + 2.5j, -0.1 + 3.0j])

lam_max, mu_min, mu_max, tmu_min, tmu_max = bf.compute_evs(moduli_test)

print(f"lambda_max   = {lam_max:.6f}")
print(f"mu_min       = {mu_min:.6f}")
print(f"mu_max       = {mu_max:.6f}")
print(f"tmu_min      = {tmu_min:.6f}")
print(f"tmu_max      = {tmu_max:.6f}")

The batched version `compute_evs_vmap(moduli_batch)` evaluates these quantities over a whole array of moduli in one vectorised call:

In [ ]:
moduli_batch = sampler.get_complex_moduli(200)
evs_batch = bf.compute_evs_vmap(jnp.array(moduli_batch))
lam_arr, mu_min_arr, mu_max_arr, tmu_min_arr, tmu_max_arr = evs_batch

print(f"lambda_max range : [{float(lam_arr.min()):.4f}, {float(lam_arr.max()):.4f}]")
print(f"mu_min    range  : [{float(mu_min_arr.min()):.4f}, {float(mu_min_arr.max()):.4f}]")
print(f"tmu_min   range  : [{float(tmu_min_arr.min()):.4f}, {float(tmu_min_arr.max()):.4f}]")

## Computing the global bounding box

`compute_bounding_box(moduli_sample)` iterates over the sample, updating the global eigenvalue extrema, and returns the $L^2$ radii of the bounding box for the NSNS-flux vector:

$$
    \|h_1\|^2 \leq \frac{N_{\max}}{s_{\min}\,\tilde\mu_{\min}^{\rm gl}}, \quad
    \|h_2\|^2 \leq \frac{N_{\max}}{s_{\min}\,\mu_{\min}^{\rm gl}}, \quad
    \|h\|^2 \leq \frac{2\,\lambda_{\max}^{\rm gl}\,N_{\max}}{s_{\min}}.
$$

The global extrema are also accessible as attributes of `bf` after the call.

In [ ]:
moduli_sample = sampler.get_complex_moduli(500)
h1_box, h2_box, h_box = bf.compute_bounding_box(moduli_sample)

print("Global eigenvalue extrema:")
print(f"  lambda_max_gl   = {bf.lambda_max_gl:.4f}")
print(f"  mu_min_gl       = {bf.mu_min_gl:.4f}")
print(f"  tilde_mu_min_gl = {bf.tilde_mu_min_gl:.4f}")
print()
print("Bounding box radii (L2 norms):")
print(f"  h1_box = {h1_box:.3f}   (|h1|^2 <= {h1_box**2:.2f})")
print(f"  h2_box = {h2_box:.3f}   (|h2|^2 <= {h2_box**2:.2f})")
print(f"  h_box  = {h_box:.3f}   (|h|^2  <= {h_box**2:.2f})")
print()
print(f"Dilaton upper bound: dil_max = {bf.dil_max:.4f}")

### Pre-computing eigenvalue bounds

For large-scale scans, computing eigenvalue bounds separately (once, with many moduli points) is more efficient than recomputing them inside each `enumerate_fluxes` or `sample_bounded_fluxes` call. The `compute_eigenvalue_bounds` method does this and stores the results as class attributes:

In [ ]:
# Pre-compute once with a large sample — reused by all subsequent calls
bf.reset_eigenvalue_bounds()  # clear any previous bounds
h1_box, h2_box, h_box = bf.compute_eigenvalue_bounds(n_sample=10_000)
print(f"\nbounds_initialized = {bf.bounds_initialized}")

The box dimensions can always be retrieved later:

In [ ]:
h1_box, h2_box, h_box = bf.get_h_box()
print(h1_box, h2_box, h_box)

## Checking bounds for a given flux

`check_bounds(moduli, tau, flux)` evaluates all eigenvalue inequalities of [arXiv:2501.03984](https://arxiv.org/abs/2501.03984) for a specific flux configuration.
It first calls `update_local` to compute the local eigenvalues and flux sub-vector norms, then evaluates every `bound_*` method.

Let us demonstrate this with a sample flux configuration:

In [ ]:
# Draw a random initial guess from the sampler
seed = 42
rns_key = jvc.PRNGSequence(seed)

sampler.update_interior_points(num_pts=5000)
moduli_0, tau_0, flux_0 = sampler.initial_guesses(1, rns_key=rns_key)
moduli_0 = moduli_0[0]
tau_0    = complex(tau_0[0])
flux_0   = flux_0[0]

print("moduli =", moduli_0)
print("tau    =", tau_0)
print("flux   =", flux_0)

In [ ]:
results = bf.check_bounds(moduli_0, tau_0, flux_0)

print(f"{'Bound':<20} {'Result'}")
print("-" * 40)
for result, label in results:
    print(f"{label:<20} {result}")

The local eigenvalue state is now stored in `bf` and can be inspected directly:

In [ ]:
print(f"s          = {bf.s:.4f}")
print(f"c0         = {bf.c0:.4f}")
print(f"N_flux     = {bf.Nflux:.4f}")
print(f"lambda_max = {bf.lambda_max:.4f}")
print(f"mu_min     = {bf.mu_min:.4f}")
print(f"tmu_min    = {bf.tilde_mu_min:.4f}")
print()
print(f"||h||^2  = {bf.hnorm:.4f}")
print(f"||f||^2  = {bf.fnorm:.4f}")
print(f"||h1||^2 = {bf.h1norm:.4f}")
print(f"||h2||^2 = {bf.h2norm:.4f}")

Individual bounds can also be queried separately without re-computing local state. Each returns a `(result, label)` tuple where `result` is a `bool` or a tuple of bools:

In [ ]:
print("h1 local : ", bf.bound_h1_local())
print("h1 global: ", bf.bound_h1_global())
print("h2 local : ", bf.bound_h2_local())
print("h2 global: ", bf.bound_h2_global())
print("h  local : ", bf.bound_h_local())
print("h  global: ", bf.bound_h_global())
print("s  local : ", bf.bound_s_local())
print("s  global: ", bf.bound_s_global())
print("f1 local : ", bf.bound_f1_local())
print("f2 local : ", bf.bound_f2_local())
print("f  local : ", bf.bound_f_local())
print("f  global: ", bf.bound_f_global())

`check_bounds_flat()` aggregates all bounds into a single pass/fail flag (no arguments — uses the current local state):

In [ ]:
all_pass, detailed = bf.check_bounds_flat()
print(f"All bounds satisfied: {all_pass}")

## Flux utility helpers

The class provides several helpers to decompose flux vectors into their sub-components.

In [ ]:
# Split flux = [f | h] into f and h (each of length n_fluxes = 2*(h12+1))
f, h = bf.get_fh(flux_0)
print("f =", f)
print("h =", h)

# Further split f = [f1 | f2] and h = [h1 | h2] (each of length dimension_H3 = h12+1)
h1, h2 = bf.get_flux_split(h)
f1, f2 = bf.get_flux_split(f)
print("h1 =", h1, "  h2 =", h2)
print("f1 =", f1, "  f2 =", f2)

# Or all at once:
h1, h2, f1, f2 = bf.get_subvector(flux_0)

# D3-tadpole charge
print(f"N_flux = {bf.get_nflux(flux_0):.4f}")

## Full enumeration algorithm

`enumerate_fluxes()` is the main entry point that combines all steps above. It:

1. Samples `n_sample` moduli points from the sampler and computes the global bounding box.
   If `compute_eigenvalue_bounds()` was called beforehand, this step is skipped.
2. Enumerates all integer $h$ candidates inside the box.
3. For each $h$, computes an ISD-projected $f$ at up to `n_isd_per_h` moduli points.
4. Retains flux configurations that satisfy the D3-tadpole constraint and all local eigenvalue bounds.

**New features (since v2):**
- **`compute_eigenvalue_bounds(n_sample)`** — pre-compute and cache eigenvalue bounds.
- **`moduli_regions`** — stratified ISD starting points via Cartesian product across moduli dimensions.
- **`use_linearised_shifts=True`** — iterative ISD refinement with flag-based early stopping (requires `FluxVacuaFinder`).
- **`constraints`** — user-supplied constraint function `(moduli, tau, flux) → bool`.
- **`chunk_size`** — override the default streaming chunk size.
- **`KeyboardInterrupt`** — gracefully returns partial results found so far.

This requires a sampler to have been passed at initialisation.

In [ ]:
# Re-initialise with a tighter tadpole to keep the enumeration fast
bf_enum = bounded_fluxes(model, sampler=sampler, Nmax=4)

# Pre-compute eigenvalue bounds (reused inside enumerate_fluxes)
bf_enum.compute_eigenvalue_bounds(n_sample=5_000)

valid_fluxes = bf_enum.enumerate_fluxes(
                            n_sample=100,
                            n_isd_per_h=10,
                            verbose=True,
                        )

## Inspecting enumerated flux vacua

The returned list contains NumPy arrays `[f | h]` of shape `(2 * n_fluxes,)`. Let us compute basic statistics:

In [ ]:
if len(valid_fluxes) > 0:
    flux_mat = np.stack(valid_fluxes)   # shape (N_valid, 2*n_fluxes)

    # D3-tadpole for each valid flux
    tadpoles = np.array([
        bf_enum.get_nflux(jnp.array(fv)) for fv in valid_fluxes
    ])

    print(f"Valid flux configurations found: {len(valid_fluxes)}")
    print(f"N_flux range : [{tadpoles.min():.0f}, {tadpoles.max():.0f}]")
    print(f"N_flux mean  : {tadpoles.mean():.2f}")
    print()
    print("First three valid flux vectors [f | h]:")
    for fv in valid_fluxes[:3]:
        print(" ", fv.astype(int))
else:
    print("No valid flux vectors found — try increasing Nmax or the moduli sample size.")

In [ ]:
import matplotlib.pyplot as plt

if len(valid_fluxes) > 0:
    fig, axes = plt.subplots(1, 2, dpi=150, figsize=(8, 3))

    axes[0].hist(tadpoles, bins=range(0, int(tadpoles.max()) + 2), edgecolor='k', linewidth=0.5)
    axes[0].set_xlabel(r"$N_{\rm flux}$", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title("D3-tadpole distribution", fontsize=11)

    # h1 vs h2 sub-vector norms
    h1norms = np.array([
        bf_enum.compute_norm(bf_enum.get_flux_split(bf_enum.get_fh(jnp.array(fv))[1])[0])
        for fv in valid_fluxes
    ])
    h2norms = np.array([
        bf_enum.compute_norm(bf_enum.get_flux_split(bf_enum.get_fh(jnp.array(fv))[1])[1])
        for fv in valid_fluxes
    ])

    axes[1].scatter(h1norms, h2norms, s=15, alpha=0.7)
    axes[1].set_xlabel(r"$\|h_1\|^2$", fontsize=12)
    axes[1].set_ylabel(r"$\|h_2\|^2$", fontsize=12)
    axes[1].set_title(r"NSNS sub-vector norms", fontsize=11)
    h1b, h2b, _ = bf_enum.get_h_box()
    axes[1].axvline(h1b**2, color='r', linestyle='--', linewidth=0.8, label=r"$h_1$ box")
    axes[1].axhline(h2b**2, color='b', linestyle='--', linewidth=0.8, label=r"$h_2$ box")
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

## Summary

The `bounded_fluxes` class provides a systematic and provably complete approach to enumerate Type IIB SUSY flux vacua in a finite region of moduli space. The key workflow is:

1. **Initialise**: `bf = bounded_fluxes(model, sampler=sampler, Nmax=...)`
2. **Compute global bounds**: `bf.compute_eigenvalue_bounds(n_sample=100_000)` — establishes the bounding box from a large moduli sample.  Stored as class attributes and reused by subsequent calls.
3. **Enumerate candidates**: `bf.enumerate_fluxes(...)` — runs the complete enumeration pipeline.
4. **Stochastic search**: `bf.sample_bounded_fluxes(...)` — randomly samples $h$-vectors for large boxes ([notebook 10b](10b_stochastic_flux_search.ipynb)).

**New features:**
| Feature | Description |
|---|---|
| `compute_eigenvalue_bounds()` | Pre-compute bounds once with large sample |
| `moduli_regions` | Cartesian product of Im(z) bands for ISD starting points |
| `use_linearised_shifts` | Iterative ISD refinement with flag-based early stopping |
| `constraints` | User-supplied constraint function |
| `chunk_size` | Override streaming chunk size |
| `KeyboardInterrupt` | Graceful return of partial results |

For further details on the mathematical derivation of the bounds see [arXiv:2501.03984](https://arxiv.org/abs/2501.03984).